In [25]:
import pandas as pd
movies = pd.read_csv("movies.csv")

In [26]:
movies

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
62418,209157,We (2018),Drama
62419,209159,Window of the Soul (2001),Documentary
62420,209163,Bad Poems (2018),Comedy|Drama
62421,209169,A Girl Thing (2001),(no genres listed)


In [148]:
#cleaning movie title 
import re
def clean_title(title):
    return re.sub("[^a-zA-z0-9 ]","",title)

In [28]:
movies["clean_title"] = movies["title"].apply(clean_title)

In [153]:
movies

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji 1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale 1995
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II 1995
...,...,...,...,...
62418,209157,We (2018),Drama,We 2018
62419,209159,Window of the Soul (2001),Documentary,Window of the Soul 2001
62420,209163,Bad Poems (2018),Comedy|Drama,Bad Poems 2018
62421,209169,A Girl Thing (2001),(no genres listed),A Girl Thing 2001


In [151]:
#turning title into a number using tfidf
from sklearn.feature_extraction.text import TfidfVectorizer #sklearn=python machine learning library
vectorizer = TfidfVectorizer (ngram_range=(1,2))
tfidf = vectorizer.fit_transform(movies["clean_title"])

In [157]:
#compute similarity using cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search(title):
    title = clean_title(title)
    query_vec = vectorizer.transform([title])
    similarity = cosine_similarity(query_vec, tfidf).flatten()
    indices = np.argpartition(similarity, -5)[-5:]
    results = movies.iloc[indices].iloc[::-1]
    return results

In [161]:
#build interactive search box
import ipywidgets as widgets
from IPython.display import display

movie_input = widgets.Text(
    value=" ",
    description="Movie Title:",
    disabled=False 
)
movie_list = widgets.Output()

def on_type(data):
    with movie_list:
        movie_list.clear_output()
        title = data["new"]
        if len(title) > 5:
            display(search(title))

movie_input.observe(on_type, names='value')
display(movie_input, movie_list)

Text(value=' ', description='Movie Title:')

Output()

In [183]:
#import ratings
ratings = pd.read_csv("ratings.csv")

In [163]:
ratings

,userId,movieId,rating,timestamp
0,1,296,5.0,1.147880e+09
1,1,306,3.5,1.147869e+09
2,1,307,5.0,1.147869e+09
3,1,665,5.0,1.147879e+09
4,1,899,3.5,1.147869e+09
...,...,...,...,...
11790412,76463,4954,4.0,1.105386e+09
11790413,76463,4958,3.5,1.105386e+09
11790414,76463,4963,4.0,1.105385e+09
11790415,76463,4975,4.0,1.105384e+09


In [164]:
ratings.dtypes

userId         int64
movieId        int64
rating       float64
timestamp    float64
dtype: object

In [165]:
movie_id =1

In [184]:
#finding users who liked the same movie
similar_users = ratings[(ratings["movieId"]==movie_id) & (ratings["rating"] >=5)]["userId"].unique()

In [167]:
similar_users

array([   36,    75,    86, ..., 76409, 76413, 76460])

In [186]:
similar_user_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"]>=4)]["movieId"]

In [187]:
similar_user_recs

5101             1
5104            11
5105            34
5106            46
5108            60
             ...  
11789804     91658
11789805     99114
11789806    112552
11789807    112556
11789808    134130
Name: movieId, Length: 835857, dtype: int64

In [188]:
similar_user_recs = similar_user_recs.value_counts() / len(similar_users)

similar_user_recs = similar_user_recs[similar_user_recs >.1]

In [189]:
similar_user_recs

movieId
1       1.000000
318     0.520636
260     0.508192
356     0.504726
296     0.459357
          ...   
2080    0.102079
8874    0.101764
2542    0.100977
2683    0.100977
899     0.100504
Name: count, Length: 254, dtype: float64

In [190]:
#how many users like movies
all_users = ratings[(ratings["movieId"].isin(similar_user_recs.index) & (ratings["rating"] >4))]

In [198]:
all_users

,userId,movieId,rating,timestamp
0,1,296,5.0,1.147880e+09
19,1,2692,5.0,1.147869e+09
29,1,4973,4.5,1.147869e+09
48,1,7361,5.0,1.147880e+09
72,2,110,5.0,1.141417e+09
...,...,...,...,...
11790141,76463,1625,4.5,1.105992e+09
11790154,76463,1704,4.5,1.105384e+09
11790201,76463,2174,4.5,1.105384e+09
11790244,76463,2571,5.0,1.105384e+09


In [191]:
all_users_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

In [199]:
all_users_recs

movieId
318     0.332527
296     0.275506
2571    0.236463
356     0.227344
593     0.217870
          ...   
440     0.016432
2078    0.014327
596     0.014040
2080    0.012686
708     0.010089
Name: count, Length: 254, dtype: float64

In [200]:
#recommendation score
rec_percentages = pd.concat([similar_user_recs, all_users_recs],axis =1)
rec_percentages.columns = ["similar", "all"]


In [197]:
rec_percentages

,similar,all
movieId,,
1,1.000000,0.121669
318,0.520636,0.332527
260,0.508192,0.213865
356,0.504726,0.227344
296,0.459357,0.275506
...,...,...
2080,0.102079,0.012686
8874,0.101764,0.044074
2542,0.100977,0.049953


In [201]:
rec_percentages["score"] = rec_percentages ["similar"] / rec_percentages["all"]

In [202]:
rec_percentages = rec_percentages.sort_values("score", ascending=False)

In [203]:
rec_percentages

,similar,all,score
movieId,,,
708,0.107435,0.010089,10.648771
2355,0.251890,0.024184,10.415787
1,1.000000,0.121669,8.218989
596,0.114524,0.014040,8.157094
440,0.133428,0.016432,8.119895
...,...,...,...
5618,0.118620,0.083282,1.424317
79132,0.180529,0.128423,1.405742
7361,0.135003,0.099277,1.359866


In [180]:
rec_percentages.head(10).merge(movies, left_index=True, right_on="movieId")

,similar,all,score,movieId,title,genres,clean_title
693,0.107435,0.010089,10.648771,708,"Truth About Cats & Dogs, The (1996)",Comedy|Romance,Truth About Cats Dogs The 1996
2264,0.251890,0.024184,10.415787,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,Bugs Life A 1998
0,1.000000,0.121669,8.218989,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
588,0.114524,0.014040,8.157094,596,Pinocchio (1940),Animation|Children|Fantasy|Musical,Pinocchio 1940
435,0.133428,0.016432,8.119895,440,Dave (1993),Comedy|Romance,Dave 1993
1991,0.102079,0.012686,8.046342,2080,Lady and the Tramp (1955),Animation|Children|Comedy|Romance,Lady and the Tramp 1955
1,0.133585,0.016610,8.042500,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji 1995
721,0.169975,0.021477,7.914377,736,Twister (1996),Action|Adventure|Romance|Thriller,Twister 1996
1989,0.110113,0.014327,7.685770,2078,"Jungle Book, The (1967)",Animation|Children|Comedy|Musical,Jungle Book The 1967
637,0.214713,0.028052,7.654027,648,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller,Mission Impossible 1996


In [204]:
def find_similar_movies(movie_id):
    similar_users = ratings[(ratings["movieId"] == movie_id) & (ratings["rating"] > 4)]["userId"].unique()
    similar_user_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"] > 4)]["movieId"]
    
    similar_user_recs = similar_user_recs.value_counts() / len(similar_users)
    similar_user_recs = similar_user_recs[similar_user_recs > .10]
    
    all_users = ratings[(ratings["movieId"].isin(similar_user_recs.index)) & (ratings["rating"] > 4)]
    all_users_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())
    
    rec_percentages = pd.concat([similar_user_recs, all_users_recs], axis=1)
    rec_percentages.columns = ["similar", "all"]
    rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]
    rec_percentages = rec_percentages.sort_values("score", ascending=False)
    
    return rec_percentages.head(10).merge(movies, left_index=True, right_on="movieId")[["score", "title", "genres"]]

In [208]:
all_genres = set()
movies["genres"].str.split("|").apply(all_genres.update)
all_genres = sorted(all_genres)

# Widgets
movie_name_input = widgets.Text(
    value=" ",
    description="Movie Title:",
    disabled=False
)
genre_dropdown = widgets.Dropdown(
    options=["All"] + all_genres,
    value="All",
    description="Genre:"
)
recommendation_list = widgets.Output()

def on_type(data):
    with recommendation_list:
        recommendation_list.clear_output()
        title = movie_name_input.value
        selected_genre = genre_dropdown.value

        if len(title) > 5:
            results = search(title)
            movie_id = results.iloc[0]["movieId"]
            recs = find_similar_movies(movie_id)
            if selected_genre != "All":
                recs = recs[recs["genres"].str.contains(selected_genre, na=False)]
            display(recs)

        elif selected_genre != "All":
            genre_movies = movies[movies["genres"].str.contains(selected_genre, na=False)]
            display(genre_movies[["title", "genres"]].head(10))

movie_name_input.observe(on_type, names="value")
genre_dropdown.observe(on_type, names="value")

display(movie_name_input, genre_dropdown, recommendation_list)

Text(value=' ', description='Movie Title:')

Dropdown(description='Genre:', options=('All', '(no genres listed)', 'Action', 'Adventure', 'Animation', 'Chil…

Output()